<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/ETF_Swing_BackTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install ta pandas_ta
!pip install --upgrade backtrader

In [43]:
import yfinance as yf
import pandas as pd
import numpy as np
import logging
import backtrader as bt
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.optimize import minimize
from scipy.stats import zscore

%matplotlib inline
print("Libraries Installed!")

Libraries Installed!


## Create Class for Backtesting

In [65]:

# Define the ETFRotationStrategy class with the required methods
class ETFRotationStrategy(bt.Strategy):
    top_n = 3 # Number of top ETFs to select and allocate equally
    rebalance_interval = 21 #~ ~63 trading days for a quartely rebalance

    def __init__(self):
        self.last_rebalance = None
        self.top_etfs = []
        self.initial_value = None
        self.portfolio_values = []  # Store portfolio values over time
        self.performance_dict = {}  # Initialize an empty dictionary to store ETF performance
        self.ema = bt.indicators.ExponentialMovingAverage(self.data.close, period=200)
        self.crossover = bt.indicators.CrossOver(self.data.close, self.ema)
        #self.ema8 = {data._name: bt.indicators.EMA(data.close, period=8) for data in self.datas}
        #self.ema13 = {data._name: bt.indicators.EMA(data.close, period=13) for data in self.datas}
        #self.sma = bt.indicators.SimpleMovingAverage(self.data.close, period=200)
        #self.crossover = bt.indicators.CrossOver(self.data.close, self.sma)

    def start(self):
      # Set the initial portfolio value
      self.initial_value = self.broker.get_value()
      print(f"\nInitial Portfolio Value: {self.initial_value}")

    def getdatabyname(self, ticker):
        """Find the corresponding data feed for the given ticker."""
        try:
            for data in self.datas:
                if data._name == ticker:
                    return data
        except Exception as e:
            print(f"Error getting data for {ticker}: {e}")
        return None

    def calculate_performance(self, ticker):
        """Calculate cisutom score using 1M, 3M, 6M performance."""
        try:
            data = self.getdatabyname(ticker)
            if data is None:
                print(f"No data found for {ticker}")
                return None
            if len(data) < 126:
                print(f"Not enough data for {data._name}")
                return None

            # Retrieve the historical close prices
            close_1w = data.close.get(size=5)
            close_1m = data.close.get(size=21)
            close_3m = data.close.get(size=63)
            close_6m = data.close.get(size=126)

             # Ensure sufficient data is returned
            if len(close_1m) < 21 or len(close_3m) < 63 or len(close_6m) < 126:
              print(f"Not enough data to calculate z-scores for {data._name}")
              return None

            # Calculate raw performance
            perf_1w = close_1w[-1] / close_1w[0] - 1
            perf_1m = close_1m[-1] / close_1m[0] - 1
            perf_3m = close_3m[-1] / close_3m[0] - 1
            perf_6m = close_6m[-1] / close_6m[0] - 1

            self.performance_dict[ticker] = (perf_1w, perf_1m, perf_3m, perf_6m)
            #print(f"ETF: {ticker}, Score: {perf_1w}")  # Debugging line
            return (perf_1w, perf_1m, perf_3m, perf_6m)
        except Exception as e:
            print(f"Error calculating score for {ticker}: {e}")
            return None

    def calculate_score(self):
      """ claculaets zscore for etfs"""
      try:
        # Extract performances for all ETFs
        perf_1w_list = [v[0] for v in self.performance_dict.values()]
        perf_1m_list = [v[1] for v in self.performance_dict.values()]
        perf_3m_list = [v[2] for v in self.performance_dict.values()]
        perf_6m_list = [v[3] for v in self.performance_dict.values()]

        # Compute the mean and std across all ETFs
        mean_1w, std_1w = np.mean(perf_1w_list), np.std(perf_1w_list)
        mean_1m, std_1m = np.mean(perf_1m_list), np.std(perf_1m_list)
        mean_3m, std_3m = np.mean(perf_3m_list), np.std(perf_3m_list)
        mean_6m, std_6m = np.mean(perf_6m_list), np.std(perf_6m_list)

        # Avoid divison by zeros
        std_1w = std_1w if std_1w > 0 else 1e-10
        std_1m = std_1m if std_1m > 0 else 1e-10
        std_3m = std_3m if std_3m > 0 else 1e-10
        std_6m = std_6m if std_6m > 0 else 1e-10

        # Compute Z-score for each ETF
        z_scores = {}
        for ticker, (perf_1w, perf_1m, perf_3m, perf_6m) in self.performance_dict.items():
            zscore_1w = (perf_1w - mean_1w) /std_1w
            zscore_1m = (perf_1m - mean_1m) /std_1m
            zscore_3m = (perf_3m - mean_3m) /std_3m
            zscore_6m = (perf_6m - mean_6m)/std_6m

            # ✅ Weighted score (adjust weights as needed)
            final_score = (0.05 * zscore_1w) + (0.6 * zscore_1m) + \
                          (0.35 * zscore_3m) + (0.* zscore_6m)
            #print(f"ETF: {ticker}, Score: {final_score}")  # Debugging line
            z_scores[ticker] = final_score
        return z_scores
      except Exception as e:
        print(f"Error calculating score for {ticker}: {e}")
        return None

    def rebalance_portfolio(self):
      try:
          print("Rebalance portfolio called.")  # Log method call
          # Close existing positions
          for data in self.datas:
            position = self.getposition(data)
            #print(f"Checking position for {data._name}: {position.size} shares")  # Debug position size
            if position.size > 0:
                self.close(data)
                print(f"Closed position for {data._name}")

          # Get the current portfolio value
          portfolio_value = self.broker.getvalue()
          print(f"Current portfolio value: {portfolio_value:.2f}")

          # Allocate portfolio equally among top ETFs
          allocation = portfolio_value / len(self.top_etfs)
          print(f"Allocation per ETF: {allocation:.2f}")

          for ticker in self.top_etfs:
            data = self.getdatabyname(ticker)
            if data:
                price = data.close[0]
                if price > 0:
                    shares = int(allocation / price)
                    self.buy(data=data, size=shares)
                    print(f"Bought {shares} shares of {ticker} at {price:.2f}")
                else:
                    print(f"Skipping {ticker} due to invalid price: {price}")
            else:
                print(f"No data feed found for {ticker}. Skipping.")
      except Exception as e:
        print(f"Error in rebalance_portfolio: {e}")


    def rebalance_portfolios(self):
      try:
          print("Rebalance portfolio called.")  # Log method call

          current_positions =  {data._name for data, pos in self.positions.items() if pos.size > 0}
          new_positions = set(self.top_etfs)

          # ✅ Identify ETFs to sell
          etfs_to_sell = current_positions - new_positions
          for etf in etfs_to_sell:
            data = self.getdatabyname(etf)
            if data:
                self.close(data)
                #print(f"Sold {etf} - No longer in top ETFs")

          # Get the current portfolio value
          portfolio_value = self.broker.getvalue()
          print(f"Current portfolio value: {portfolio_value:.2f}")
          # ✅ Identify ETFs to buy
          etfs_to_buy = new_positions - current_positions
          if etfs_to_buy:
              cash_per_etf = self.broker.get_cash() / len(etfs_to_buy)  # Equal allocation
              for etf in etfs_to_buy:
                 data = self.getdatabyname(etf)
                 if data:
                  size = int(cash_per_etf / data.close[0])
                  self.buy(data=data, size=size)
                  #print(f"Bought {etf} - Newly added to top ETFs")

          print(f"\nRebalanced Portfolio on {self.datetime.date()}:")
          print(f"Kept Holdings: {current_positions & new_positions}")
          print(f"Sold: {etfs_to_sell}")
          print(f"Bought: {etfs_to_buy}")
      except Exception as e:
        print(f"Error in rebalance_portfolio: {e}")



    def stop(self):
      """ Stop method """
      try:
        # Calculate the final portfolio value
        final_value = self.broker.get_value()

        # Ensure initial portfolio value is stored
        if not hasattr(self, 'initial_value'):
          print("Error: Initial portfolio value not set!")
          return

        # Calculate the growth percentage
        growth_percentage = ((final_value - self.initial_value) / self.initial_value) * 100

        # Display the results
        print(f"Initial Portfolio Value: {self.initial_value:.2f}")
        print(f"Final Portfolio Value: {final_value:.2f}")
        print(f"Portfolio Growth: {growth_percentage:.2f}%")

        # Plot the portfolio growth
        dates, values = zip(*self.portfolio_values)  # Unpack dates and values
        plt.figure(figsize=(10, 6))
        plt.plot(dates, values, label='Portfolio Value', color='blue')
        plt.xlabel('Date')
        plt.ylabel('Portfolio Value')
        plt.title('Portfolio Growth Over Time')
        plt.legend()
        plt.grid()
        plt.show()

      except Exception as e:
        print(f"Error in stop: {e}")

    def next(self):
      try:
        # Debugging: Print available data names to check if they are strings or tuples
        #print("Available data names:", self.getdatanames())

        # Skip the warm-up period (first 126 trading days)
        if len(self.datas[0]) <126:
          return # Do nothing until enough data is available

        # Record the initial value at the start of the backtest
        #if self.initial_value is None:
            #self.initial_value = self.broker.get_value()

        # Record the portfolio value at each step
        current_value = self.broker.getvalue()
        self.portfolio_values.append((self.datetime.date(), current_value))

        # Exit Positions if Price Drops Below EMA Before Rebalancing
        for data, position in self.positions.items():
          if  position.size > 0:
            ticker = data._name
            # Calculate EMA(200) for this ETF
            if self.crossover < 0:  # Price crosses below EMA
              self.close(data)
              print(f"Exiting {ticker} on {self.datetime.date()} (Crossed Below EMA)")


        # Rebalance portfolo every quarter
        if self.last_rebalance is None or (self.datetime.date() - self.last_rebalance).days >= self.rebalance_interval:
          self.last_rebalance = self.datetime.date()

          # Get the performance for each ETF
          for data in self.datas:
            ticker = data._name
            self.calculate_performance(ticker)

          # Calculate z-scores
          z_scores = self.calculate_score()
          if not z_scores:
            print("No valid z-scores found")
            return

          # Get the SPY's Z-score as the benchmark
          spy_zscore = z_scores.get('SPY', 0)
          if spy_zscore is None:
            print("SPY Z-score not found in the dictionary.")
            return

          # Compute the relative scores (ETF Z-score - SPY Z-score)
          relative_scores = {ticker: score - spy_zscore for ticker, score in z_scores.items() if ticker != "SPY"}
          # Debugging: Check relative scores
          #print(f"Relative Scores: {relative_scores}")


          if relative_scores is not None and len(relative_scores) > 0:
            #print(f"Z-scores: {z_scores}")  # Debugging line
            # You can now use relative_scores/z_scores to rank ETFs or select top performers
            sorted_etfs = sorted(relative_scores.items(), key=lambda x: x[1], reverse=True)
            self.top_etfs = [x[0] for x in sorted_etfs[:self.top_n]]
            # Log the top ETFs
            print(f"\nTop {self.top_n} ETFs to buy for {self.datetime.date()}: {self.top_etfs}")
          else:
            print("No valid relative-scores found.")
          # Rebalance portfolio
          self.rebalance_portfolio()
      except Exception as e:
        print(f"Error in next(): {e}")

# Function to set up and run the test
def run_test():
    cerebro = bt.Cerebro()
    # Set up the broker (optional but recommended)
    cerebro.broker.set_cash(10000)  # Starting cash for the backtest
    cerebro.broker.setcommission(commission=0.)  # Commission per trade
    # Set the position size to overide default sizing i.e. The number of units bought = cash available / asset price
    #cerebro.addsizer(bt.sizers.FixedSize, stake=100)

    # Create dummy data for two ETFs
    #df_o = pd.read_csv('etf_list.csv')
    #tickers = df_o['ETF'].to_list()
    tickers = ['SPY','QQQ','XLK','XLF','XLY','XLP','XLU', 'XLV','XLP','XLRE','GBTC','XLC', 'XLB','XLI',
               'EWH', 'VGK', 'EWO', 'UNG','FEZ','SOXX','SH']
    for ticker in tickers:
        df =  yf.download(ticker, start='2020-04-01', end='2025-03-01')
        df.columns = df.columns.droplevel(1)  # Drop the ticker symbol from column names
        data = bt.feeds.PandasData(dataname=df, name=ticker)
        cerebro.adddata(data)

    # Add the strategy to Cerebro
    cerebro.addstrategy(ETFRotationStrategy)

    #Run the Cerebro engine to trigger the `next` method
    cerebro.run()
    cerebro.plot()

    # Run the Cerebro engine to trigger the `next` method and retrieve the strategy instance
    #strategies = cerebro.run()  # This returns a tuple of strategy instances
    #strategy = strategies[0]  # Access the first strategy instance

    # Test `calculate_normalized_score` on the ETF list
    #for ticker in tickers:
        #score = strategy.calculate_normalized_score(ticker)
        #print(f"Calculated normalized score for {ticker}: {score}")


# Execute the test
if __name__ == "__main__":
    run_test()


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********


Initial Portfolio Value: 10000

Top 3 ETFs to buy for 2021-01-15: ['GBTC', 'EWO', 'XLF']
Rebalance portfolio called.
Current portfolio value: 10000.00
Allocation per ETF: 3333.33
Bought 84 shares of GBTC at 39.34
Bought 193 shares of EWO at 17.19
Bought 115 shares of XLF at 28.77

Top 3 ETFs to buy for 2021-02-05: ['XLRE', 'XLC', 'UNG']
Rebalance portfolio called.
Closed position for GBTC
Closed position for EWO
Current portfolio value: 9741.02
Allocation per ETF: 3247.01
Bought 97 shares of XLRE at 33.17
Bought 47 shares of XLC at 68.81
Bought 77 shares of UNG at 42.12

Top 3 ETFs to buy for 2021-02-26: ['GBTC', 'XLF', 'SOXX']
Rebalance portfolio called.
Closed position for XLRE
Closed position for XLC
Closed position for UNG
Current portfolio value: 10108.38
Allocation per ETF: 3369.46
Bought 77 shares of GBTC at 43.20
Bought 112 shares of XLF at 30.05
Bought 25 shares of SOXX at 134.06

Top 3 ETFs to buy for 2021-03-19: ['XLF', 'GBTC', 'XLI']
Rebalance portfolio called.
Closed posi

<IPython.core.display.Javascript object>